In [1]:
import pandas as pd
import numpy as np
import os
import re
import json
os.chdir('..')

In [2]:
from typing import List, Dict, Literal

In [3]:
splits = ['train', 'dev', 'test']
# splits = ['train']
lang = 'indo'
dataset_type = 'hoasa_aug'
dataset_per_split = {}
for split in splits:
    with open(f'dataset/{dataset_type}/{lang}/mvp_aos_raw/{split}.json', 'r') as f:
        dataset_per_split[split] = json.load(f)
print(splits)

['train', 'dev', 'test']


In [4]:
df_per_split = {}
for split in splits:
    df_per_split[split] = pd.DataFrame(dataset_per_split[split])

In [5]:
def add_space_around_punctuation(text):
    # Except for '-'
    # Ensure space before punctuation
    text = re.sub(r'(\S)([.,!?\(\)\"\';:+/]+)', r'\1 \2', text)
    # Ensure space after punctuation
    text = re.sub(r'([.,!?\(\)\"\';:+/]+)(\S)', r'\1 \2', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    # Ensure punctuation sequences like '...' are split into spaced dots
    text = re.sub(r'([.]{1,})', lambda m: ' '.join(m.group(1)), text)
    # Ensure punctuation sequences like '!!' or other punctuations that appear consecutively are split into spaced characters
    text = re.sub(r'([!,?\(\)]{1,})', lambda m: ' '.join(m.group(1)), text)
    return text.strip()

def parse_absa_string(text: str):
    """
    Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
    Each dictionary contains the tag as the key and the corresponding value.
    For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
    [{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
    {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

    Args:
        text (str): ABSA string output to be parsed.

    Returns:
        List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

    """
    pattern = r"\[(\w+)\]\s*([^[]+)"
    matches = re.findall(pattern, text)

    result = []
    current_dict = {}

    for tag, content in matches:
        if tag == "SSEP":  # Sentence separator -> Start a new dictionary
            result.append(current_dict)
            current_dict = {}
        else:
            current_dict[tag] = content.strip()

    if current_dict:  # Append the last sentence if it exists
        result.append(current_dict)

    return result

def convert_to_absa_format(triplets: List[Dict[str, str]], order: List[Literal['A', 'O', 'S']]) -> str:
	"""
	Converts a list of dictionaries containing ABSA triplets into a formatted string.
	Each dictionary should contain keys 'A', 'O', and 'S' for Aspect, Opinion, and Sentiment respectively.
	The order of these elements in the output string is determined by the 'order' parameter.

	Args:
		triplets (List[Dict[str, str]]): List of dictionaries with ABSA triplet information.
	Returns:
		str: A formatted string representing the ABSA triplets.
	"""
	result = []
	for triplet in triplets:
		parts = []
		for key in order:
			if key in triplet:
				parts.append(f"[{key}] {triplet[key]}")
		result.append(" ".join(parts))
	return " [SSEP] ".join(result)

def lower_triplets(triplets: List[Dict[str, str]]) -> List[Dict[str, str]]:
    return [{k: v.lower() for k, v in triplet.items()} for triplet in triplets]

def punctuation_triplets(triplets: List[Dict[str, str]]) -> List[Dict[str, str]]:
	return [{k: add_space_around_punctuation(v) for k, v in triplet.items()} for triplet in triplets]

In [6]:
from copy import deepcopy
def check_if_target_in_input(row):
	input_sentence = deepcopy(row['input_temp'])
	target_triplets = deepcopy(row['target_temp'])
	for triplet in target_triplets:
		aspect = triplet['A']
		opinion = triplet['O']
		if (aspect not in input_sentence or opinion not in input_sentence) and (aspect != 'null' and opinion != 'null'):
			print(f"Aspect or opinion not found in input {row['sentence_id']}: Aspect='{aspect}', Opinion='{opinion}'")
			print(f"Input sentence: {input_sentence}")

In [7]:
for split in splits:
	df_per_split[split]['target_temp'] = df_per_split[split]['target'].apply(lambda x: punctuation_triplets(lower_triplets(parse_absa_string(x))))
	df_per_split[split]['input_temp'] = df_per_split[split]['input'].apply(lambda x: f"{add_space_around_punctuation(x.split('[A] [O] [S]')[0]).lower()} [A] [O] [S]")
	df_per_split[split].apply(check_if_target_in_input, axis=1)
	df_per_split[split]['target_temp'] = df_per_split[split]['target_temp'].apply(lambda x: convert_to_absa_format(x, order=['A', 'O', 'S']))

In [8]:
new_dataset_per_split = {}
instance_id = 0
for split in splits:
    new_dataset_per_split[split] = []
    for index, row in df_per_split[split].iterrows():
        new_dataset_per_split[split].append({
            'sentence_id': row['sentence_id'],
            'instance_id': instance_id,
            'input': deepcopy(row['input_temp']),
            'target': deepcopy(row['target_temp']),
            'element_order': 'aos',
            'task_elements': 'aos'
        })
        instance_id += 1

In [10]:
dataset_type

'hoasa_aug'

In [11]:
# Write to json file
dataset_path = f'dataset/{dataset_type}/{lang}/mvp_aos'
os.makedirs(dataset_path, exist_ok=True)
for split, data in new_dataset_per_split.items():
    with open(f"{dataset_path}/{split}.json", "w") as f:
        json.dump(data, f, indent=4)